In [ ]:
import moku_util as mu

# The moku_util module loads these libraries for internal methods,
# but you may want to run alternative code that uses them directly.
import matplotlib.pyplot as plt
import numpy as np

#### Load the data and look at some raw traces

As noted in the lab manual, you'll want to have data from **both SiPM slow channels** to facilitate the following analysis

In [ ]:
# Acquired data from both 'slow' channels of the SiPM readout, to serve
# as example. You'll probably want to acquire your own data and change
# the filename appropriately.
#
# Note that the file itself is too large to be hosted on GitHub, but 
# is available on Canvas for download. You shouldn't need to use it
# since you can acquire your own data, but it's there.
filename = '../example_data/cs137_2channel_pulses.li'
df = mu.DataFile(filename)

#### Plot the data and zoom in on various features to see if the logged data matches your expectation

Compare what you see to what you've been observing on the oscilloscope.

In [ ]:
%matplotlib widget
fig, ax = df.plot_data(input=[0,1], microseconds=True, xlim=(0,10000))

#### Make sure to close the figure for the later parts of this notebook!

You won't be able to interact with the figure and zoom in etc if you close the figure prematurely, so hopefully you didn't just use the "Run All" feature of the notebook...

In [ ]:
plt.close(fig)

---
#### The function executed from below scapes the data and finds peaks above baseline

Data from the peak finding, such as peak locations etc are stored as class attributes. The raw output from `scipy.signal.find_peaks()` is stored, as well as some derived attributes. 

There are some default values, but you should change these based on your own observations of the baseline and peak height.

In [ ]:
# The relevant input parameters are:
#     - the channel to analyze
#     - the minimum height of a peak, i.e. a threshold value
#     - the minimum width of a peak in units of samples
#     - the prominence of a peak, i.e. the necessary height
#       of the peak above baseline to be considered as a peak.
#       The tuple is for minimum and maximum prominence, although
#       the maximum is not used in this example.
#     - a boolean to specify whether to look for negative peaks,
#       where all the other inputs should still remain positive
#       but be interpreted as negative values.
df.find_peaks(
    channel=0, 
    height=0.5, 
    width=2, 
    prominence=(0.3, None),
    negative_signal=False
)

#### To do spectroscopy, we need to know the energy associated to a particular pulse

To estimate deposited energy, we can integrate the peaks. This works because the area under the amplified pulses are proportional to the current/charge output from the SiPMs, which is proportional to the number of photons detected. 

To understand this, first note that the number of scintillation photons produced is proportional to the energy deposited by ionizing radiation and characterized by a _light yield_, which you can obtain from a manufacturer datasheet. A subset of these photons get detected by the SiPMs, converted to electrons, and multiplied by the avalanche gain of the device. Assuming the charge $Q$ from the SiPM deposits onto the capacitor more-or-less instantaneously, the voltage on the output of the first amplifier is thus,

$$ V_{\rm fast} \approx - \frac{Q}{C_{\rm in}} e^{- t / (R_{\rm in} C_{\rm in})} $$

where the minus sign comes from some details about which way the current is flowing in the circuit, but is somewhat unimportant for this analysis. Following the passive low-pass filter and subsequent amplification, the voltage at the output of the slow amplifier is roughly,

$$ V_{\rm slow} \approx A \frac{Q}{C_{\rm in}} (1 - e^{- t / (R_{\rm LPF} C_{\rm LPF})}) e^{- t / (R_{\rm out} C_{\rm out})} $$

where the additional minus sign (for an overall positive result) is from the inverting topology, $A$ is a gain factor, and the various exponential terms come from the $RC$ charging and discharging inherent in the readout circuit. If we integrate this pulse over a timescale longer than a few $e$-foldings of all the exponential terms, the result is proportional to the prefactor, i.e.,

$$ \int V_{\rm slow} \, dt \propto Q \propto E_\gamma $$

where we've dropped a bunch of dimensionally relevant proportionality factors that are quite complex to account for exactly.

##### Big picture: the integral of the pulse is proportional to gamma ray energy

In [ ]:
# The integrate_peaks() method loops over all of the peaks found
# with the previous function, and integrates the area under the pulse
# for each peak. The pre_peak and post_peak parameters specify how many
# samples to include before and after the peak, respectively, in order
# capture the tails of the full pulse that are below the threshold.
#
# Start with the default value and adjust after checking below
df.integrate_peaks(
    channel=0, 
    pre_peak=3, 
    post_peak=8
)

#### Let's show the peak integration window for a few pulses

This allows us to check the pre- and post-peak values and adjust as necessary.

In [ ]:
# Change the backend to inline for quick plotting of the peak
# integrations. The plot_peak_integrations() method allows many 
# plots to be generated one after another, which only really works
# with the inline backend.
#
# The plots will include vertical lines to indicate the location of 
# the pulse maximum, as well as horizontal lines indicate the baseline
# which is calculated automatically without user input. The pulse
# area should be shaded in blue.
%matplotlib inline
df.plot_peak_integrations(channel=0, nplot=1) # Change nplot to show more sets

#### If the high-lighted peak area doesn't look right visually (e.g. not getting the full pulse, or getting too much of the baseline), adjust the `pre_peak` and `post_peak` options in the `integrate_peaks()` function and iterate until it looks good.

---
#### Let's take a look at the results of our integration

In the plotting below, you'll need to change the index of the `peak_integrals` array to match the channel you chose to analyze

In [ ]:
# Set the number of bins for a histogram of the pulse integrals.
# This will be the primary parameter you can adjust, although you 
# may also want to generate a bin_edges array manually once you
# know what's happening.
nbins = 100

# Set the bin_edges dynamically using the max and min of the 
# pulse integrals and the number of bins you specified
bin_edges = np.linspace(
    df.peak_integrals[0].min(), 
    df.peak_integrals[0].max(), 
    nbins+1
)

# Alternatively, you could use logarithmic spacing for the bins,
# but leave this commented out for now
# bin_edges = np.logspace(
#     np.log10(df.peak_integrals[0].min()), 
#     np.log10(df.peak_integrals[0].max()), 
#     nbins+1
# )

# Plot the data and extract the histogram values in one go
%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df.peak_integrals[0], bins=bin_edges)
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
# ax.set_yscale('log') # You may wish to use a logarithmic scale
# ax.set_xscale('log')
fig.tight_layout()
plt.show()

#### Adjust the definition of `bin_edges` to optimize the display of your histogram of pulse integrals

Note that the units of the pulse integral are unimportant, since we dropped a bunch of proportionality factors. 

#### Once you've optimized the display, interpret your results!

Since the pulse integrals are proportional to deposited energy, your histogram is essentially an uncalibrated _spectrum_ of the deposited energy from incident gamma rays! This is **spectroscopy**! Our detector apparatus produces signals that vary in response to input energy, which then allows us to distinguish events with different energies!

As you're examining your spectrum, remember that gamma rays in this energy range will primarily interact via two process: (1) photoelectric absorption wherein the full energy of the gamma ray is deposited in the scintillator, and (2) Compton scattering wherein the gamma ray will deposit some portion of it's energy via an inelastic collision, with the amount of energy determined by the kinematics of the collision itself.

##### What do you see in your spectrum? Can you relate any features to the above two processes?

In [ ]:
# Close all open figures to avoid memory leaks, since you probably 
# executed the previous plot a few times while adjusting parameters.
plt.close('all')

---
### Now let's build the same spectrum for the background dataset

Remember how we collected some data without the source present? Let's analyze that data!

In [ ]:
# Acquired data from both 'slow' channels of the SiPM readout, to serve
# as example. You'll probably want to acquire your own data and change
# the filename appropriately.
background_filename = '../example_data/background_2channel_pulses.li'
background_df = mu.DataFile(background_filename)

Make sure to change all the function parameters/options below to match those used in your analysis of the Cs-137 pulses. You may also need to change the `channel` argument to select the right data. 

Hopefully you have some good notes about the various experimental conditions and channels in the data you saved!

In [ ]:
background_df.find_peaks(
    channel=0, 
    height=0.5, 
    width=2, 
    prominence=(0.3, None),
    negative_signal=False
)

background_df.integrate_peaks(
    channel=0, 
    pre_peak=3, 
    post_peak=8
)

In the plotting below, you'll need to change the index of the `peak_integrals` array to match the channel you chose to analyze. You'll need to do this for both DataFiles individually.

In [ ]:
# Plot the background and signal spectra together to compare
%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df.peak_integrals[0], bins=bin_edges)
ax.hist(background_df.peak_integrals[0], bins=bin_edges)
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
# ax.set_yscale('log') # You may wish to use a logarithmic scale
# ax.set_xscale('log')
fig.tight_layout()
plt.show()

#### Compare the two spectra!

How does the background spectrum differ from the spectrum with the source present?

Are there any identifiable features in the background spectrum? If so, can you explain them? If not, can you explain why?

---
### Synthesizing everything you've learned from this notebook, consider if you want to take more data with different experimental conditions or total integration times.

#### If you want to take more data, do that now!